In [2]:
pip install git+https://github.com/openai/CLIP.git


Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-5nc7xnje
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-5nc7xnje
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369488 sha256=c597c1885156e820e40c497dd5084ff8ecff9ac0dac31daddf73368ce4fafdb2
  Stored in directory: /tmp/pip-ephem-wheel-cache-4ec_vs5g/wheels/ab/4f/3a/5e51521b55997aa6f0690e095c08824219753128ce8d9969a3
Successfully built clip
Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
build_prototypes_dlrsd.py

Build CLIP image prototypes for classes from one-channel masks (DLRSD).
Saves prototypes to prototypes_dlrsd.pt

Requirements:
  pip install git+https://github.com/openai/CLIP.git
  pip install pillow numpy torch torchvision tqdm

Edit params below if needed.
"""

import os
import random
import torch
import clip
import numpy as np
from PIL import Image
from tqdm import tqdm

# -------------------------
# USER CONFIG (already provided)
# -------------------------
IMAGE_DIR = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images'
MASK_DIR  = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_1cmasks'

CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

# Patch + prototype params
NUM_CLASSES = 17           # classes to build prototypes for (1..NUM_CLASSES)
PATCHES_PER_CLASS = 40     # how many patches to collect per class
PATCH_SIZE = 128           # square patch size (pixels)
CLIP_MODEL = "ViT-B/32"    # CLIP model name (OpenAI repo)
OUTPUT_PROTOTYPES = "prototypes_dlrsd.pt"

# randomness control
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -------------------------
# helpers
# -------------------------
def list_image_files(folder):
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
    files = [f for f in sorted(os.listdir(folder)) if f.lower().endswith(exts)]
    return files


def stem(filename):
    return os.path.splitext(filename)[0]


def build_mask_map(mask_dir):
    """Return dict: stem -> filename for masks."""
    files = list_image_files(mask_dir)
    m = {stem(f): f for f in files}
    return m


def load_mask_as_ids(mask_path):
    """
    Loads a single-channel mask assuming pixel values are class IDs (0..17).
    Returns numpy int32 array HxW.
    """
    m = Image.open(mask_path).convert("L")
    arr = np.array(m, dtype=np.int32)
    return arr


def extract_patch_centered_on_class(img_pil, mask_np, class_id, patch_size=PATCH_SIZE, max_tries=30):
    """
    Randomly sample pixels of class_id and return a square PIL patch centered on that pixel
    if ≥30% of the patch pixels belong to class_id. Otherwise try up to max_tries times.
    Returns PIL.Image or None.
    """
    H, W = mask_np.shape
    ys, xs = np.where(mask_np == class_id)
    if len(xs) == 0:
        return None

    half = patch_size // 2
    for _ in range(max_tries):
        idx = random.randint(0, len(xs) - 1)
        cx, cy = xs[idx], ys[idx]

        x0 = cx - half
        y0 = cy - half
        x1 = cx + half
        y1 = cy + half

        if x0 < 0 or y0 < 0 or x1 >= W or y1 >= H:
            continue

        patch_mask = mask_np[y0:y1, x0:x1]
        # require a minimum fraction of pixels be of class_id
        if (patch_mask == class_id).sum() < (patch_size * patch_size) * 0.30:
            continue

        patch_img = img_pil.crop((x0, y0, x1, y1))
        return patch_img

    return None


# -------------------------
# main builder
# -------------------------
def build_prototypes(image_dir=IMAGE_DIR, mask_dir=MASK_DIR,
                     num_classes=NUM_CLASSES, patches_per_class=PATCHES_PER_CLASS,
                     patch_size=PATCH_SIZE, clip_model=CLIP_MODEL, out_file=OUTPUT_PROTOTYPES):
    # sanity checks
    if not os.path.isdir(image_dir):
        raise FileNotFoundError(f"Image dir not found: {image_dir}")
    if not os.path.isdir(mask_dir):
        raise FileNotFoundError(f"Mask dir not found: {mask_dir}")

    print("Device:", DEVICE)
    print("Loading CLIP model:", clip_model)
    model, preprocess = clip.load(clip_model, device=DEVICE)
    model.eval()

    image_files = list_image_files(image_dir)
    mask_map = build_mask_map(mask_dir)

    if len(image_files) == 0:
        raise RuntimeError("No images found in image_dir")

    # prepare storage
    proto_feats = {cid: [] for cid in range(1, num_classes + 1)}

    # iterate images and collect patches
    print(f"Found {len(image_files)} images. Collecting patches...")
    for img_name in tqdm(image_files):
        key = stem(img_name)
        if key not in mask_map:
            # try minor variants: sometimes masks have different suffixes
            # skip if not found
            # print warning
            print(f"WARNING: no matching mask for image {img_name} (expected stem {key}) -- skipping")
            continue

        img_path = os.path.join(image_dir, img_name)
        mask_path = os.path.join(mask_dir, mask_map[key])

        img_pil = Image.open(img_path).convert("RGB")
        mask_np = load_mask_as_ids(mask_path)

        for cid in range(1, num_classes + 1):
            if len(proto_feats[cid]) >= patches_per_class:
                continue
            patch = extract_patch_centered_on_class(img_pil, mask_np, cid, patch_size)
            if patch is None:
                continue
            # preprocess & encode
            patch_t = preprocess(patch).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feat = model.encode_image(patch_t)   # shape (1, D)
                feat = feat / feat.norm(dim=-1, keepdim=True)
            proto_feats[cid].append(feat.cpu())

    # average and normalize
    final_protos = {}
    for cid in range(1, num_classes + 1):
        feats = proto_feats[cid]
        if len(feats) == 0:
            print(f"WARNING: collected 0 patches for class {cid} ({CLASS_INFO.get(cid, {}).get('name','?')})")
            continue
        stacked = torch.cat(feats, dim=0)  # (N, D)
        mean_vec = stacked.mean(dim=0)
        mean_vec = mean_vec / mean_vec.norm()
        final_protos[cid] = mean_vec

        print(f"Class {cid} ({CLASS_INFO.get(cid, {}).get('name','?')}): collected {stacked.shape[0]} patches")

    # save
    torch.save(final_protos, out_file)
    print("Saved prototypes to:", out_file)
    return final_protos


if __name__ == "__main__":
    build_prototypes()


Device: cuda
Loading CLIP model: ViT-B/32



00%|███████████████████████████████████████| 338M/338M [00:04<00:00, 73.8MiB/s]

Found 630 images. Collecting patches...


100%|█████████████████████████████████████████| 630/630 [00:36<00:00, 17.14it/s]

Class 1 (airplane): collected 2 patches
Class 2 (bare soil): collected 40 patches
Class 3 (buildings): collected 40 patches
Class 4 (cars): collected 29 patches
Class 5 (chaparral): collected 30 patches
Class 6 (court): collected 31 patches
Class 8 (field): collected 26 patches
Class 9 (grass): collected 40 patches
Class 10 (mobile home): collected 23 patches
Class 11 (pavement): collected 40 patches
Class 12 (sand): collected 36 patches
Class 13 (sea): collected 30 patches
Class 14 (ship): collected 29 patches
Class 15 (tanks): collected 26 patches
Class 16 (trees): collected 40 patches
Class 17 (water): collected 40 patches
Saved prototypes to: prototypes_dlrsd.pt


In [1]:
import os
import numpy as np
from PIL import Image

MASK_DIR = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_1cmasks'

unique_values = set()

for fname in os.listdir(MASK_DIR):
    if not fname.lower().endswith((".png", ".jpg", ".tif", ".tiff")):
        continue

    path = os.path.join(MASK_DIR, fname)
    arr = np.array(Image.open(path))

    # If mask has 3 channels, show a warning
    if arr.ndim == 3:
        print(f"WARNING: mask {fname} is RGB, shape {arr.shape}")
        # convert to tuple classes so we know the RGB triplets
        vals = {tuple(v) for v in arr.reshape(-1,3)}
        unique_values.update(vals)
    else:
        # single-channel mask
        vals = np.unique(arr)
        unique_values.update(vals.tolist())

print("\n====== UNIQUE MASK VALUES FOUND ======")
print(sorted(unique_values))
print("======================================")



====== UNIQUE MASK VALUES FOUND ======
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [2]:
import torch
import clip
import numpy as np
from PIL import Image
import os
import torch.nn.functional as F
from collections import defaultdict
import json

# Your class information
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

class CLIPPrototypeBuilder:
    def __init__(self, model_name="ViT-B/32"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model, self.preprocess = clip.load(model_name, device=self.device)
        print(f"Loaded CLIP model: {model_name} on {self.device}")
        
    def extract_image_features(self, image_path):
        """Extract CLIP features for a single image"""
        try:
            image = Image.open(image_path).convert('RGB')
            image_input = self.preprocess(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                image_features = self.model.encode_image(image_input)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                
            return image_features.cpu().numpy()
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
    
    def extract_class_prototypes(self, images_dir, masks_dir, class_info):
        """
        Extract class prototypes from training images using CLIP
        
        Args:
            images_dir: Directory containing training images
            masks_dir: Directory containing dense annotated masks  
            class_info: CLASS_INFO dictionary with class mappings
        """
        # Get all image files
        image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        
        # Initialize storage for class features
        class_features = defaultdict(list)
        class_patch_counts = defaultdict(int)
        
        print(f"Processing {len(image_files)} training images...")
        
        for idx, img_file in enumerate(image_files):
            if idx % 50 == 0:
                print(f"Processing image {idx}/{len(image_files)}...")
                
            img_path = os.path.join(images_dir, img_file)
            
            # Load corresponding mask
            mask_file = img_file.replace('.jpg', '.png').replace('.jpeg', '.png')
            mask_path = os.path.join(masks_dir, mask_file)
            
            if not os.path.exists(mask_path):
                # Try other extensions
                for ext in ['.png', '.jpg', '.jpeg']:
                    mask_file = img_file.split('.')[0] + ext
                    mask_path = os.path.join(masks_dir, mask_file)
                    if os.path.exists(mask_path):
                        break
            
            if not os.path.exists(mask_path):
                print(f"Mask not found for {img_file}")
                continue
            
            # Extract image features
            img_features = self.extract_image_features(img_path)
            if img_features is None:
                continue
                
            # Load and process mask
            mask = np.array(Image.open(mask_path))
            
            # For multi-class: create patches for each class present
            unique_classes = np.unique(mask)
            
            for class_idx in unique_classes:
                if class_idx in class_info:  # Valid class index
                    class_name = class_info[class_idx]["name"]
                    
                    # Create binary mask for this class
                    class_mask = (mask == class_idx)
                    
                    if np.any(class_mask):
                        class_features[class_name].append(img_features)
                        class_patch_counts[class_name] += 1
        
        # Compute prototypes (mean features) for each class
        class_prototypes = {}
        prototype_info = {}
        
        print("\n=== CLIP Class Prototype Collection Results ===")
        for class_idx, info in class_info.items():
            class_name = info["name"]
            features_list = class_features.get(class_name, [])
            if len(features_list) > 0:
                # Stack all features and compute mean
                all_features = np.vstack(features_list)
                class_prototypes[class_name] = np.mean(all_features, axis=0, keepdims=True)
                patch_count = class_patch_counts[class_name]
                print(f"Class {class_idx} ({class_name}): collected {patch_count} patches")
                prototype_info[class_name] = {
                    'class_idx': class_idx,
                    'prototype': class_prototypes[class_name],
                    'patch_count': patch_count,
                    'feature_dim': class_prototypes[class_name].shape[1],
                    'rgb': info["rgb"]
                }
            else:
                print(f"WARNING: collected 0 patches for class {class_idx} ({class_name})")
                class_prototypes[class_name] = None
                prototype_info[class_name] = {
                    'class_idx': class_idx,
                    'prototype': None,
                    'patch_count': 0,
                    'feature_dim': 0,
                    'rgb': info["rgb"]
                }
        
        return class_prototypes, prototype_info
    
    def save_prototypes(self, prototypes, prototype_info, save_path="clip_prototypes_dlrsd.pt"):
        """Save prototypes in PyTorch format"""
        save_data = {
            'prototypes': prototypes,
            'prototype_info': prototype_info,
            'class_info': CLASS_INFO,  # Save the original CLASS_INFO
            'model_type': 'CLIP',
            'feature_dim': 512  # CLIP ViT-B/32 feature dimension
        }
        
        torch.save(save_data, save_path)
        print(f"Saved prototypes to: {save_path}")
        
        # Also save as JSON for readability
        json_info = {}
        for class_name, info in prototype_info.items():
            json_info[class_name] = {
                'class_idx': info['class_idx'],
                'patch_count': info['patch_count'],
                'feature_dim': info['feature_dim'],
                'rgb': info['rgb'],
                'prototype_shape': info['prototype'].shape if info['prototype'] is not None else None
            }
        
        with open(save_path.replace('.pt', '_info.json'), 'w') as f:
            json.dump(json_info, f, indent=2)
        
        return save_path

    def compute_similarity(self, query_features, class_prototypes):
        """Compute similarity between query features and class prototypes"""
        similarities = {}
        
        for class_name, prototype in class_prototypes.items():
            if prototype is not None:
                # Cosine similarity
                query_norm = query_features / np.linalg.norm(query_features)
                prototype_norm = prototype / np.linalg.norm(prototype)
                similarity = np.dot(query_norm.flatten(), prototype_norm.flatten())
                similarities[class_name] = similarity
        
        return similarities

# Usage for your DLRSD dataset
def main():
    # Initialize CLIP prototype builder
    clip_builder = CLIPPrototypeBuilder()
    
    # Paths to your training data
    images_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images'
    masks_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_1cmasks'
    
    # Extract class prototypes using CLASS_INFO
    class_prototypes, prototype_info = clip_builder.extract_class_prototypes(
        images_dir, masks_dir, CLASS_INFO
    )
    
    # Save prototypes
    save_path = clip_builder.save_prototypes(class_prototypes, prototype_info)
    
    print(f"\nPrototypes saved successfully!")
    print(f"Total classes processed: {len(CLASS_INFO)}")
    
    return class_prototypes, prototype_info

if __name__ == "__main__":
    prototypes, info = main()

Loaded CLIP model: ViT-B/32 on cuda
Processing 630 training images...
Processing image 0/630...
Processing image 50/630...
Processing image 100/630...
Processing image 150/630...
Processing image 200/630...
Processing image 250/630...
Processing image 300/630...
Processing image 350/630...
Processing image 400/630...
Processing image 450/630...
Processing image 500/630...
Processing image 550/630...
Processing image 600/630...

=== CLIP Class Prototype Collection Results ===
Class 1 (airplane): collected 30 patches
Class 2 (bare soil): collected 208 patches
Class 3 (buildings): collected 211 patches
Class 4 (cars): collected 271 patches
Class 5 (chaparral): collected 34 patches
Class 6 (court): collected 32 patches
Class 7 (dock): collected 30 patches
Class 8 (field): collected 26 patches
Class 9 (grass): collected 308 patches
Class 10 (mobile home): collected 30 patches
Class 11 (pavement): collected 377 patches
Class 12 (sand): collected 67 patches
Class 13 (sea): collected 31 patche

In [3]:
import torch
import clip
import numpy as np
from PIL import Image
import os
import torch.nn.functional as F
from collections import defaultdict
import json

CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

class CLIPPseudoLabelRefiner:
    def __init__(self, prototype_path, model_name="ViT-B/32"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model, self.preprocess = clip.load(model_name, device=self.device)
        
        # Load prototypes
        prototype_data = torch.load(prototype_path, map_location=self.device)
        self.class_prototypes = prototype_data['prototypes']
        self.prototype_info = prototype_data['prototype_info']
        
        print(f"Loaded CLIP model and {len(self.class_prototypes)} class prototypes")
    
    def extract_region_features(self, image, mask, class_idx):
        """Extract CLIP features for a specific class region in the image"""
        # Create binary mask for the class
        class_mask = (mask == class_idx)
        
        if not np.any(class_mask):
            return None
            
        # Get bounding box of the region
        rows = np.any(class_mask, axis=1)
        cols = np.any(class_mask, axis=0)
        ymin, ymax = np.where(rows)[0][[0, -1]]
        xmin, xmax = np.where(cols)[0][[0, -1]]
        
        # Add padding
        pad = 10
        h, w = mask.shape
        xmin = max(0, xmin - pad)
        xmax = min(w, xmax + pad)
        ymin = max(0, ymin - pad)
        ymax = min(h, ymax + pad)
        
        # Crop the region
        region = image.crop((xmin, ymin, xmax, ymax))
        
        # Extract features
        region_input = self.preprocess(region).unsqueeze(0).to(self.device)
        with torch.no_grad():
            region_features = self.model.encode_image(region_input)
            region_features = region_features / region_features.norm(dim=-1, keepdim=True)
        
        return region_features.cpu().numpy()
    
    def compute_class_similarity(self, region_features, class_name):
        """Compute similarity between region features and class prototype"""
        if class_name not in self.class_prototypes or self.class_prototypes[class_name] is None:
            return 0.0
            
        prototype = self.class_prototypes[class_name]
        
        # Cosine similarity
        region_norm = region_features / np.linalg.norm(region_features)
        prototype_norm = prototype / np.linalg.norm(prototype)
        similarity = np.dot(region_norm.flatten(), prototype_norm.flatten())
        
        return similarity
    
    def refine_pseudo_mask(self, image_path, pseudo_mask_path, confidence_threshold=0.6):
        """Refine pseudo-mask using CLIP class prototypes"""
        # Load image and pseudo-mask
        image = Image.open(image_path).convert('RGB')
        pseudo_mask = np.array(Image.open(pseudo_mask_path))
        
        refined_mask = pseudo_mask.copy()
        confidence_map = np.zeros_like(pseudo_mask, dtype=np.float32)
        correction_log = []
        
        # Get unique classes in the pseudo-mask
        unique_classes = np.unique(pseudo_mask)
        
        for class_idx in unique_classes:
            if class_idx == 0:  # Skip background
                continue
                
            class_name = CLASS_INFO[class_idx]["name"]
            
            # Skip if no prototype for this class
            if class_name not in self.class_prototypes or self.class_prototypes[class_name] is None:
                print(f"Warning: No prototype for class {class_idx} ({class_name})")
                continue
            
            # Extract features for this class region
            region_features = self.extract_region_features(image, pseudo_mask, class_idx)
            if region_features is None:
                continue
            
            # Compute similarity with its own class
            own_similarity = self.compute_class_similarity(region_features, class_name)
            
            # Compute similarities with all other classes
            other_similarities = {}
            for other_idx, other_info in CLASS_INFO.items():
                if other_idx == 0 or other_idx == class_idx:
                    continue
                other_similarity = self.compute_class_similarity(region_features, other_info["name"])
                other_similarities[other_idx] = other_similarity
            
            # Find best matching class
            best_class = class_idx
            best_similarity = own_similarity
            
            for other_idx, other_sim in other_similarities.items():
                if other_sim > best_similarity:
                    best_similarity = other_sim
                    best_class = other_idx
            
            # Apply correction if confidence is low or wrong class
            if best_class != class_idx and best_similarity > confidence_threshold:
                # Update the mask
                refined_mask[pseudo_mask == class_idx] = best_class
                correction_log.append({
                    'original_class': class_idx,
                    'new_class': best_class,
                    'confidence': best_similarity,
                    'region_size': np.sum(pseudo_mask == class_idx)
                })
                print(f"Corrected: {CLASS_INFO[class_idx]['name']} -> {CLASS_INFO[best_class]['name']} (conf: {best_similarity:.3f})")
            
            # Store confidence for this class region
            confidence_map[pseudo_mask == class_idx] = own_similarity
        
        return refined_mask, confidence_map, correction_log
    
    def batch_refine_pseudo_masks(self, images_dir, pseudo_masks_dir, output_dir, confidence_threshold=0.6):
        """Refine all pseudo-masks in batch"""
        os.makedirs(output_dir, exist_ok=True)
        confidence_dir = os.path.join(output_dir, "confidence_maps")
        os.makedirs(confidence_dir, exist_ok=True)
        
        image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        
        total_corrections = 0
        refinement_report = []
        
        print(f"Refining {len(image_files)} pseudo-masks...")
        
        for idx, img_file in enumerate(image_files):
            if idx % 50 == 0:
                print(f"Processing {idx}/{len(image_files)}...")
            
            image_path = os.path.join(images_dir, img_file)
            
            # Find corresponding pseudo-mask
            mask_file = img_file.replace('.jpg', '.png').replace('.jpeg', '.png')
            pseudo_mask_path = os.path.join(pseudo_masks_dir, mask_file)
            
            if not os.path.exists(pseudo_mask_path):
                print(f"Pseudo-mask not found for {img_file}")
                continue
            
            # Refine the pseudo-mask
            refined_mask, confidence_map, corrections = self.refine_pseudo_mask(
                image_path, pseudo_mask_path, confidence_threshold
            )
            
            # Save refined mask
            output_path = os.path.join(output_dir, mask_file)
            Image.fromarray(refined_mask.astype(np.uint8)).save(output_path)
            
            # Save confidence map
            confidence_path = os.path.join(confidence_dir, mask_file)
            confidence_uint8 = (confidence_map * 255).astype(np.uint8)
            Image.fromarray(confidence_uint8).save(confidence_path)
            
            total_corrections += len(corrections)
            refinement_report.append({
                'image': img_file,
                'corrections': corrections,
                'total_corrections': len(corrections)
            })
        
        # Save refinement report
        report_path = os.path.join(output_dir, "refinement_report.json")
        with open(report_path, 'w') as f:
            json.dump(refinement_report, f, indent=2)
        
        print(f"\n=== CLIP Pseudo-label Refinement Complete ===")
        print(f"Total images processed: {len(image_files)}")
        print(f"Total class corrections: {total_corrections}")
        print(f"Refined masks saved to: {output_dir}")
        print(f"Confidence maps saved to: {confidence_dir}")
        print(f"Detailed report: {report_path}")
        
        return refinement_report

# Usage example
def main():
    # Initialize the refiner with your saved prototypes
    refiner = CLIPPseudoLabelRefiner("clip_prototypes_dlrsd.pt")
    
    # Paths to your data
    images_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_images"  # Images for pseudo-labels
    pseudo_masks_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/topo_refined_predictions'  # Initial SAM pseudo-masks
    output_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_refined_predictions'
    
    # Refine all pseudo-masks
    report = refiner.batch_refine_pseudo_masks(
        images_dir=images_dir,
        pseudo_masks_dir=pseudo_masks_dir,
        output_dir=output_dir,
        confidence_threshold=0.6  # Adjust based on your needs
    )
    
    return report

if __name__ == "__main__":
    report = main()

/tmp/ipykernel_20659/3695187377.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prototype_data = torch.load(prototype_path, map_location=self.device)


Loaded CLIP model and 18 class prototypes
Refining 1319 pseudo-masks...
Processing 0/1319...
Corrected: trees -> field (conf: 0.901)
Corrected: buildings -> airplane (conf: 0.917)
Corrected: cars -> airplane (conf: 0.917)
Corrected: pavement -> airplane (conf: 0.917)
Corrected: grass -> airplane (conf: 0.865)
Corrected: pavement -> airplane (conf: 0.865)
Corrected: cars -> airplane (conf: 0.855)
Corrected: pavement -> airplane (conf: 0.855)
Corrected: buildings -> field (conf: 0.844)
Corrected: cars -> airplane (conf: 0.905)
Corrected: pavement -> airplane (conf: 0.905)
Corrected: bare soil -> field (conf: 0.854)
Corrected: buildings -> airplane (conf: 0.915)
Corrected: cars -> airplane (conf: 0.939)
Corrected: pavement -> airplane (conf: 0.910)
Corrected: bare soil -> ship (conf: 0.829)
Corrected: buildings -> ship (conf: 0.802)
Corrected: cars -> airplane (conf: 0.928)
Corrected: grass -> airplane (conf: 0.904)
Corrected: pavement -> airplane (conf: 0.917)
Corrected: sand -> ship (co

TypeError: Object of type uint8 is not JSON serializable

In [4]:
import torch
import clip
import numpy as np
from PIL import Image
import os
import torch.nn.functional as F
from collections import defaultdict

CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

class CLIPPseudoLabelRefiner:
    def __init__(self, prototype_path, model_name="ViT-B/32"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model, self.preprocess = clip.load(model_name, device=self.device)
        
        # Load prototypes
        prototype_data = torch.load(prototype_path, map_location=self.device)
        self.class_prototypes = prototype_data['prototypes']
        self.prototype_info = prototype_data['prototype_info']
        
        print(f"Loaded CLIP model and {len(self.class_prototypes)} class prototypes")
    
    def extract_region_features(self, image, mask, class_idx):
        """Extract CLIP features for a specific class region in the image"""
        # Create binary mask for the class
        class_mask = (mask == class_idx)
        
        if not np.any(class_mask):
            return None
            
        # Get bounding box of the region
        rows = np.any(class_mask, axis=1)
        cols = np.any(class_mask, axis=0)
        ymin, ymax = np.where(rows)[0][[0, -1]]
        xmin, xmax = np.where(cols)[0][[0, -1]]
        
        # Add padding
        pad = 10
        h, w = mask.shape
        xmin = max(0, xmin - pad)
        xmax = min(w, xmax + pad)
        ymin = max(0, ymin - pad)
        ymax = min(h, ymax + pad)
        
        # Crop the region
        region = image.crop((xmin, ymin, xmax, ymax))
        
        # Extract features
        region_input = self.preprocess(region).unsqueeze(0).to(self.device)
        with torch.no_grad():
            region_features = self.model.encode_image(region_input)
            region_features = region_features / region_features.norm(dim=-1, keepdim=True)
        
        return region_features.cpu().numpy()
    
    def compute_class_similarity(self, region_features, class_name):
        """Compute similarity between region features and class prototype"""
        if class_name not in self.class_prototypes or self.class_prototypes[class_name] is None:
            return 0.0
            
        prototype = self.class_prototypes[class_name]
        
        # Cosine similarity
        region_norm = region_features / np.linalg.norm(region_features)
        prototype_norm = prototype / np.linalg.norm(prototype)
        similarity = np.dot(region_norm.flatten(), prototype_norm.flatten())
        
        return similarity
    
    def refine_pseudo_mask(self, image_path, pseudo_mask_path, confidence_threshold=0.6):
        """Refine pseudo-mask using CLIP class prototypes"""
        # Load image and pseudo-mask
        image = Image.open(image_path).convert('RGB')
        pseudo_mask = np.array(Image.open(pseudo_mask_path))
        
        refined_mask = pseudo_mask.copy()
        confidence_map = np.zeros_like(pseudo_mask, dtype=np.float32)
        
        # Get unique classes in the pseudo-mask
        unique_classes = np.unique(pseudo_mask)
        
        for class_idx in unique_classes:
            if class_idx == 0:  # Skip background
                continue
                
            class_name = CLASS_INFO[class_idx]["name"]
            
            # Skip if no prototype for this class
            if class_name not in self.class_prototypes or self.class_prototypes[class_name] is None:
                print(f"Warning: No prototype for class {class_idx} ({class_name})")
                continue
            
            # Extract features for this class region
            region_features = self.extract_region_features(image, pseudo_mask, class_idx)
            if region_features is None:
                continue
            
            # Compute similarity with its own class
            own_similarity = self.compute_class_similarity(region_features, class_name)
            
            # Compute similarities with all other classes
            other_similarities = {}
            for other_idx, other_info in CLASS_INFO.items():
                if other_idx == 0 or other_idx == class_idx:
                    continue
                other_similarity = self.compute_class_similarity(region_features, other_info["name"])
                other_similarities[other_idx] = other_similarity
            
            # Find best matching class
            best_class = class_idx
            best_similarity = own_similarity
            
            for other_idx, other_sim in other_similarities.items():
                if other_sim > best_similarity:
                    best_similarity = other_sim
                    best_class = other_idx
            
            # Apply correction if confidence is low or wrong class
            if best_class != class_idx and best_similarity > confidence_threshold:
                # Update the mask
                refined_mask[pseudo_mask == class_idx] = best_class
                print(f"Corrected: {CLASS_INFO[class_idx]['name']} -> {CLASS_INFO[best_class]['name']} (conf: {best_similarity:.3f})")
            
            # Store confidence for this class region
            confidence_map[pseudo_mask == class_idx] = own_similarity
        
        return refined_mask, confidence_map
    
    def batch_refine_pseudo_masks(self, images_dir, pseudo_masks_dir, output_dir, confidence_threshold=0.6):
        """Refine all pseudo-masks in batch"""
        os.makedirs(output_dir, exist_ok=True)
        confidence_dir = os.path.join(output_dir, "confidence_maps")
        os.makedirs(confidence_dir, exist_ok=True)
        
        image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        
        total_corrections = 0
        
        print(f"Refining {len(image_files)} pseudo-masks...")
        
        for idx, img_file in enumerate(image_files):
            if idx % 50 == 0:
                print(f"Processing {idx}/{len(image_files)}...")
            
            image_path = os.path.join(images_dir, img_file)
            
            # Find corresponding pseudo-mask
            mask_file = img_file.replace('.jpg', '.png').replace('.jpeg', '.png')
            pseudo_mask_path = os.path.join(pseudo_masks_dir, mask_file)
            
            if not os.path.exists(pseudo_mask_path):
                print(f"Pseudo-mask not found for {img_file}")
                continue
            
            # Refine the pseudo-mask
            refined_mask, confidence_map = self.refine_pseudo_mask(
                image_path, pseudo_mask_path, confidence_threshold
            )
            
            # Save refined mask as PNG
            output_path = os.path.join(output_dir, mask_file)
            Image.fromarray(refined_mask.astype(np.uint8)).save(output_path)
            
            # Save confidence map as PNG
            confidence_path = os.path.join(confidence_dir, mask_file)
            confidence_uint8 = (confidence_map * 255).astype(np.uint8)
            Image.fromarray(confidence_uint8).save(confidence_path)
        
        print(f"\n=== CLIP Pseudo-label Refinement Complete ===")
        print(f"Total images processed: {len(image_files)}")
        print(f"Refined masks saved to: {output_dir}")
        print(f"Confidence maps saved to: {confidence_dir}")
        
        return output_dir

# Usage example
def main():
    # Initialize the refiner with your saved prototypes
    refiner = CLIPPseudoLabelRefiner("clip_prototypes_dlrsd.pt")
    
    # Paths to your data
    images_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_images"  # Images for pseudo-labels
    pseudo_masks_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/topo_refined_predictions'  # Initial SAM pseudo-masks
    output_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_refined_predictions'
    
    # Refine all pseudo-masks
    output_path = refiner.batch_refine_pseudo_masks(
        images_dir=images_dir,
        pseudo_masks_dir=pseudo_masks_dir,
        output_dir=output_dir,
        confidence_threshold=0.6  # Adjust based on your needs
    )
    
    print(f"All refined masks saved as PNG files in: {output_path}")
    return output_path

if __name__ == "__main__":
    output_path = main()

/tmp/ipykernel_20659/2134502543.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prototype_data = torch.load(prototype_path, map_location=self.device)


Loaded CLIP model and 18 class prototypes
Refining 1319 pseudo-masks...
Processing 0/1319...
Corrected: trees -> field (conf: 0.901)
Corrected: buildings -> airplane (conf: 0.917)
Corrected: cars -> airplane (conf: 0.917)
Corrected: pavement -> airplane (conf: 0.917)
Corrected: grass -> airplane (conf: 0.865)
Corrected: pavement -> airplane (conf: 0.865)
Corrected: cars -> airplane (conf: 0.855)
Corrected: pavement -> airplane (conf: 0.855)
Corrected: buildings -> field (conf: 0.844)
Corrected: cars -> airplane (conf: 0.905)
Corrected: pavement -> airplane (conf: 0.905)
Corrected: bare soil -> field (conf: 0.854)
Corrected: buildings -> airplane (conf: 0.915)
Corrected: cars -> airplane (conf: 0.939)
Corrected: pavement -> airplane (conf: 0.910)
Corrected: bare soil -> ship (conf: 0.829)
Corrected: buildings -> ship (conf: 0.802)
Corrected: cars -> airplane (conf: 0.928)
Corrected: grass -> airplane (conf: 0.904)
Corrected: pavement -> airplane (conf: 0.917)
Corrected: sand -> ship (co

In [5]:
import numpy as np
from PIL import Image
import os

CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

def label_to_rgb(mask):
    """Convert label mask to RGB mask using CLASS_INFO colors"""
    h, w = mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    for class_idx, info in CLASS_INFO.items():
        rgb_mask[mask == class_idx] = info["rgb"]
    
    return rgb_mask

def convert_masks_to_rgb(input_dir, output_dir):
    """Convert all masks in input_dir to RGB and save in output_dir"""
    os.makedirs(output_dir, exist_ok=True)
    
    mask_files = [f for f in os.listdir(input_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"Converting {len(mask_files)} masks to RGB...")
    
    for mask_file in mask_files:
        # Load the mask
        mask_path = os.path.join(input_dir, mask_file)
        mask = np.array(Image.open(mask_path))
        
        # Convert to RGB
        rgb_mask = label_to_rgb(mask)
        
        # Save RGB mask
        output_path = os.path.join(output_dir, mask_file)
        Image.fromarray(rgb_mask).save(output_path)
        
        if len(mask_files) <= 10 or mask_files.index(mask_file) % 100 == 0:
            print(f"Converted: {mask_file}")
    
    print(f"\n=== Conversion Complete ===")
    print(f"Input: {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Total masks converted: {len(mask_files)}")

# Usage - just update these paths
if __name__ == "__main__":
    input_directory = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_refined_predictions"  # Your corrected masks
    output_directory = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_rgb_masks"  # New folder for RGB
    
    convert_masks_to_rgb(input_directory, output_directory)

Converting 1319 masks to RGB...
Converted: runwa_1890.png
Converted: runwa_1877.png
Converted: river_1389.png
Converted: harbo_494.png
Converted: build_1691.png
Converted: airpl_871.png
Converted: mediu_1269.png
Converted: airpl_857.png
Converted: airpl_887.png
Converted: baseb_949.png
Converted: golfc_67.png
Converted: golfc_40.png
Converted: parki_266.png
Converted: freew_381.png

=== Conversion Complete ===
Input: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_refined_predictions
Output: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/clip_topo_rgb_masks
Total masks converted: 1319
